# Unit 2 Assignment: Building a Mixture of Experts (MoE) Router

This notebook walks through the creation of a smart customer support router using the Groq API and a mixture of experts architecture.

## 1. Setup

In [1]:
# install dependencies
!pip install --quiet groq python-dotenv


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
from dotenv import load_dotenv
import groq

In [3]:
# load environment variables
load_dotenv()
GROQ_API_KEY = os.getenv('GROQ_API_KEY')
if not GROQ_API_KEY:
    # prompt the user directly if the key is missing
    GROQ_API_KEY = input("Enter your GROQ_API_KEY: ").strip()
    if not GROQ_API_KEY:
        raise ValueError('GROQ_API_KEY is required. Please provide it via input or set it in a .env file')

# configure the client
client = groq.Client(api_key=GROQ_API_KEY)

## 2. Define Your Experts

In [ ]:
MODEL_CONFIG = {
    'technical': {
        'system_prompt': "You are a highly technical assistant. Provide thorough, code-focused, and precise answers.",
        'temperature': 0.7
    },
    'billing': {
        'system_prompt': "You are an empathetic billing specialist. Answer with financial detail, policy guidance, and a friendly tone.",
        'temperature': 0.7
    },
    'sales': {   # ⭐ NEW expert
        'system_prompt': "You are a persuasive sales assistant. Explain product features, pricing, and benefits clearly.",
        'temperature': 0.7
    },
    'general': {
        'system_prompt': "You are a general-purpose assistant capable of casual conversation and light information.",
        'temperature': 0.7
    },
    'tool': {   # ⭐ Tool expert
        'system_prompt': "You handle tool-based queries.",
        'temperature': 0
    }
}

## 3. The Router function

In [ ]:
def route_prompt(user_input: str) -> str:
    """Classify input and detect tool-use queries."""

    text = user_input.lower()

    # ⭐ TOOL ROUTING (fast & deterministic)
    if "bitcoin" in text and ("price" in text or "current" in text):
        return "tool"

    prompt = (
        "Classify this text into one of these categories: "
        "[technical, billing, sales, general].\n"
        "Return ONLY one word.\n\n"
        f"Text: {user_input}"
    )

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        max_tokens=5
    )

    category = response.choices[0].message.content.strip().lower()

    if category not in MODEL_CONFIG:
        category = "general"

    return category

## 4. The Orchestrator

In [ ]:

def get_bitcoin_price():
    # mock API call
    return "$42,000 (mock)"

In [ ]:
def process_request(user_input: str) -> str:
    """Route input and either call a tool or an expert model."""

    category = route_prompt(user_input)

    # ⭐ SHOW ROUTING DECISION (great for demo)
    print(f"🔀 Routing → {category.upper()} expert")

    # ⭐ TOOL HANDLING
    if category == "tool":
        return f"💰 Current Bitcoin price: {get_bitcoin_price()}"

    config = MODEL_CONFIG[category]

    messages = [
        {"role": "system", "content": config["system_prompt"]},
        {"role": "user", "content": user_input}
    ]

    resp = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=messages,
        temperature=config["temperature"],
        max_tokens=300
    )

    return resp.choices[0].message.content.strip()

### Example usage

In [ ]:
queries = [
    "My python script throws an IndexError.",
    "I was charged twice this month.",
    "I want to know the pricing of your premium plan.",
    "What is the current price of Bitcoin?",
    "Tell me a joke."
]

for q in queries:
    print("Input:", q)
    print("Category:", route_prompt(q))
    print("Response:", process_request(q))
    print("-" * 60)

Input: My python script is throwing an IndexError on line 5.
Category: technical
Response: To help you resolve the `IndexError` on line 5 of your Python script, I'll need more information. 

### Debugging Steps

1. **Provide the code**: Please share the Python script, especially the lines surrounding line 5 where the error occurs. This will help identify the context and the specific issue causing the error.
2. **Error message**: Share the full error message you're seeing. This will give more details about what's going wrong.

Without seeing the code, I can only provide general guidance on how to debug an `IndexError`:

### Common Causes of IndexError

- **Out-of-range indexing**: Attempting to access an element in a list, tuple, or string with an index that does not exist (e.g., trying to access `my_list[5]` when `my_list` only has 5 elements, indexed from 0 to 4).
- **Empty sequences**: Trying to access an element from an empty list, tuple, or string.

### Example of How to Avoid Inde

## 🚀 Bonus Challenge: Tool Use Expert

The following section demonstrates adding a tool-driven expert for things like fetching the current price of Bitcoin. 

In [8]:
# simple mock tool function
def get_bitcoin_price():
    # in a real system this would call an API
    return '$42,000 (mock)'

# (continue with any additional code as necessary for the bonus challenge)
